# Paso 1: Recuperar y verificar los componentes del avance de proyecto.

## Tabla de trazabilidad — Avance de Proyecto

| Elemento | Archivo o ubicación | Versión | Estado | Acción requerida |
|---|---|---|---|---|
| Dataset | Se carga vía `kagglehub` (`blastchar/telco-customer-churn`), no hay CSV local en el repo | Consultado 25–31 ago 2026 | Disponible (requiere `kagglehub` y conexión) | Documentar que requiere autenticación/conexión a Kaggle |
| Pipeline | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 13) | ColumnTransformer con imputación (mediana), StandardScaler, OneHotEncoder, OrdinalEncoder | Corre limpio | Ninguna |
| Baseline | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 14) | DummyClassifier (most_frequent) — Acc 0.7346, Recall 0.0000 | Completo | Ninguna |
| Modelo preliminar | `Avance_Proyecto/Avance_Proyecto.ipynb` (Pasos 15–19) | Regresión Logística (max_iter=1000, solver='lbfgs') — Acc 0.8055, F1 0.6040, ROC-AUC 0.8421 | Seleccionado y justificado | Ninguna |
| Métricas | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 16) | Accuracy, Precision, Recall, F1, ROC-AUC para ambos modelos | Completo | Ninguna |
| Informe del avance de proyecto | `Avance_Proyecto/Avance_Proyecto.ipynb` (87 celdas) | Ejecutado 31 ago 2026, Python 3.14.7 | Corre de inicio a fin sin errores | Ninguna |

# Paso 2: Documentar la evolución del proyecto.

| Elemento modificado | Situación en avance de proyecto | Cambio realizado | Justificación | Impacto esperado |
|---|---|---|---|---|
| Datos | Telco Customer Churn (Kaggle, 7043 registros, 21 variables), cargado en tiempo de ejecución vía `kagglehub`. TotalCharges tenía tipo incorrecto (str) con 11 valores nulos ocultos, corregido a float64. Sin duplicados. Variable objetivo `Churn` con 26.5% de abandono (clase desbalanceada). | Sin cambios. Se verificó la calidad del dataset (valores faltantes, duplicados, tipos de datos, categorías, rango de valores, balance de clases) confirmando que conserva las condiciones del avance original. | Cumplir con la revisión de calidad de datos solicitada, asegurando que la base sobre la que se trabajará se mantiene íntegra antes de continuar con el proyecto. | Ninguno — se confirma que los datos son consistentes y confiables para las siguientes etapas (modelado, métricas, etc.). |
| Preprocesamiento | ColumnTransformer con: imputación por mediana en numéricas, StandardScaler en tenure/MonthlyCharges/TotalCharges, OneHotEncoder en categóricas nominales, OrdinalEncoder en Contract (variable ordinal). Partición 70/15/15 (train/val/test) estratificada, semilla 42. | Se detectó que `SeniorCitizen` y `PaperlessBilling` no estaban asignadas a ningún grupo del `ColumnTransformer` (pasaban sin procesar vía `remainder='passthrough'`). Se agregaron a `cols_bin` para recibir imputación y codificación explícita (`OrdinalEncoder`). También se documentó (comentario en código) la coexistencia de la partición 70/15/15 del avance con la partición 80/20 (`X_train`/`X_test`) usada en el pipeline. | La revisión de calidad de datos evidenció que esas dos variables quedaban fuera del flujo de transformación controlado, lo cual podía afectar el entrenamiento; además, era necesario aclarar la diferencia entre `x_train` y `X_train` para evitar confusión. | Ambas variables ahora pasan por imputación y codificación numérica explícita en vez de colarse sin procesar; mayor claridad y mantenibilidad del notebook, sin alterar la lógica ni los resultados obtenidos en el avance. |
| Modelos | Baseline: DummyClassifier (most_frequent). Dos modelos comparados: Regresión Logística (max_iter=1000, solver='lbfgs') y Árbol de Decisión (max_depth=6, criterion='gini'). | | | |
| Métricas | Regresión Logística: Accuracy 0.8055, Precision 0.6572, Recall 0.5588, F1 0.6040, ROC-AUC 0.8421. Árbol de Decisión: Accuracy 0.7913, Precision 0.6342, Recall 0.5053, F1 0.5625, ROC-AUC 0.8345. Baseline: Accuracy 0.7346, Recall 0.0000. | | | |
| Umbral | Umbral estándar de 0.50 evaluado junto con 0.30 y 0.70 sobre Regresión Logística. Con 0.50: Precision 0.6522, Recall 0.5893, F1 0.6191. No se generó gráfica de curva ROC explícita, solo tabla comparativa de umbrales. | | | |
| Interpretabilidad | Regresión Logística seleccionada como modelo preliminar por mejor ROC-AUC, mejor Recall (clave para el negocio), interpretabilidad de coeficientes y menor riesgo de sobreajuste frente al árbol. | | | |

In [ ]:
# --- Carga del dataset ---
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    file_path,
)

# --- Limpieza básica ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# --- Separación de variables predictoras y objetivo ---
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

# --- Partición train/val/test (70/15/15, estratificada) ---
from sklearn.model_selection import train_test_split

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

y_val_encoded = y_val  # ya viene codificado desde y

print(f"Entrenamiento: {x_train.shape}, Validación: {x_val.shape}, Prueba: {x_test.shape}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# --- 1. División de datos (tal como en el avance) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 2. Definición de columnas ---
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService']
cols_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
cols_ord = ['Contract']

# --- 3. Sub-pipelines por tipo de dato ---
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

# --- 4. Ensamblaje del ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('bin', pipe_bin, cols_bin),
        ('cat', pipe_cat, cols_cat),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='passthrough'
)

# --- 5. Ajuste (Fit) SOLO en Train; Transformación en Train y Test ---
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print("Pipeline construido y aplicado exitosamente.")
print(f"Dimensiones originales de X_train: {X_train.shape}")
print(f"Dimensiones de X_train_prep (tras One-Hot Encoding): {X_train_prep.shape}")

In [ ]:
# NOTA: Esta sección replica el Paso 13 del avance de proyecto, que usa su propia
# partición 80/20 (X_train, X_test en MAYÚSCULA) — distinta de la partición 70/15/15
# (x_train, x_val, x_test en minúscula) definida arriba para el Paso 12.
# Ambas conviven en este notebook: la de mayúscula es la que se usa para ajustar
# y evaluar el pipeline de preprocesamiento tal como estaba en el avance original.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# --- 1. División de datos (tal como en el avance) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 2. Definición de columnas ---
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 'SeniorCitizen', 'PaperlessBilling']
cols_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
cols_ord = ['Contract']

# --- 3. Sub-pipelines por tipo de dato ---
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

# --- 4. Ensamblaje del ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('bin', pipe_bin, cols_bin),
        ('cat', pipe_cat, cols_cat),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='passthrough'
)

# --- 5. Ajuste (Fit) SOLO en Train; Transformación en Train y Test ---
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print("Pipeline construido y aplicado exitosamente.")
print(f"Dimensiones originales de X_train: {X_train.shape}")
print(f"Dimensiones de X_train_prep (tras One-Hot Encoding): {X_train_prep.shape}")

# Paso 3: Revisar nuevamente la calidad de los datos.

In [ ]:
# ============================================================
# Revisión de calidad de datos (Actividad "Revisar nuevamente
# la calidad de los datos") — verifica que el dataset conserve
# las condiciones definidas en el avance de proyecto.
# ============================================================

print("="*60)
print("1. VALORES FALTANTES")
print("="*60)
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("No hay valores nulos.")

print("\n" + "="*60)
print("2. DUPLICADOS")
print("="*60)
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"customerID duplicados: {df['customerID'].duplicated().sum()}")

print("\n" + "="*60)
print("3. TIPOS DE DATOS")
print("="*60)
print(df.dtypes)

print("\n" + "="*60)
print("4. CATEGORÍAS ÚNICAS POR VARIABLE CATEGÓRICA")
print("="*60)
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

print("\n" + "="*60)
print("5. VALORES FUERA DE RANGO (numéricas)")
print("="*60)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(df[num_cols].describe())
print(f"\ntenure negativo o >72 meses: {((df['tenure']<0)|(df['tenure']>72)).sum()}")
print(f"MonthlyCharges <= 0: {(df['MonthlyCharges']<=0).sum()}")

print("\n" + "="*60)
print("6. VARIABLES EXCLUIDAS DEL MODELADO")
print("="*60)
print("customerID: identificador único, sin valor predictivo -> excluida de X")
print(f"Columnas usadas en X: {list(X.columns)}")

print("\n" + "="*60)
print("7. BALANCE DE CLASES (variable objetivo)")
print("="*60)
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(4)*100)

print("\n" + "="*60)
print("8. POSIBLE FUGA DE INFORMACIÓN (fit solo en train)")
print("="*60)
print("Verificar: ¿el preprocesador se ajustó (fit) SOLO con X_train?")
print("-> Sí: fit_transform() se aplicó únicamente sobre X_train,")
print("   y transform() (sin fit) sobre X_test. Confirmar en la celda del pipeline.")

print("\n" + "="*60)
print("9. CORRESPONDENCIA ENTRE COLUMNAS Y PIPELINE")
print("="*60)
todas_cols_pipeline = set(cols_num + cols_bin + cols_cat + cols_ord)
todas_cols_X = set(X.columns)
faltantes_en_pipeline = todas_cols_X - todas_cols_pipeline
faltantes_en_X = todas_cols_pipeline - todas_cols_X
print(f"Columnas en X pero NO asignadas a ningún grupo del pipeline: {faltantes_en_pipeline}")
print(f"Columnas en el pipeline pero que NO existen en X: {faltantes_en_X}")
if not faltantes_en_pipeline and not faltantes_en_X:
    print("Correspondencia correcta: todas las columnas de X están cubiertas por el pipeline.")

In [ ]:
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 'SeniorCitizen', 'PaperlessBilling']

Revision de la calidad de los datos:

Se verificaron los 9 puntos solicitados (valores faltantes, duplicados, tipos de datos, categorías, rango de valores, variables excluidas, balance de clases, fuga de información y correspondencia columnas-pipeline) sobre el dataset del avance de proyecto

Hallazgo y corrección: Las variables 'SeniorCitizen' y 'PaperlessBilling' no estan asignadas a ningúb grupo del 'ColumnTransformer' y pasaban sin procesar vía 'remainder='passthrough'. Se corrigió agregándolas a 'cols_bin' para que reciban imputación y codificación explicita ('OrdinalEncoder')

Resto de puntos sin hallazgos. El dataset conserva las condiciones definidas en avance de proyecto (sin nulos, sin duplicados, sin valores fuera de rango, balance de clases 73.46%/26.54% consistente con lo reportafo, y sin fuga de información entre train y test)

# Paso 4: Consolidar el pipeline de preprocesamiento.